In [11]:
import requests
import getpass
import pandas as pd
from io import StringIO
import pyspark
from pyspark.sql.functions import col
from pyspark.sql.functions import lit
from pyspark.sql import functions as F

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 15, Finished, Available, Finished, False)

In [3]:
path = "Files/bronze/reference/taxi_zone_lookup.csv"
df =(spark.read.option("header", True)).csv(path)
df.show()

StatementMeta(, 47e7fc46-9d21-4aba-aad7-d99cbed0b79a, 7, Finished, Cancelled, Cancelled, False)

Py4JJavaError: An error occurred while calling o7854.showString.
: org.apache.spark.SparkException: [SPARK_JOB_CANCELLED] Job 1 cancelled part of cancelled job group 7 SQLSTATE: XXKDA
	at org.apache.spark.errors.SparkCoreErrors$.sparkJobCancelled(SparkCoreErrors.scala:222)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3428)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$7(DAGScheduler.scala:1331)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.scala:18)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1331)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3751)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3724)
	at org.apache.spark.scheduler.DAGS

In [ ]:
print(df.dtypes)
print(df.filter(col("LocationID").isNull()).count())
print(df.select("LocationID").distinct().count())
print(df.select("LocationID").count())

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 7, Finished, Available, Finished, False)

[('LocationID', 'string'), ('Borough', 'string'), ('Zone', 'string'), ('service_zone', 'string')]
0
265
265


In [ ]:
#changing LOcation ID to int
df = df.withColumn("LocationID", col("LocationID").cast("int"))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 8, Finished, Available, Finished, False)

In [ ]:
df.write.format("delta").mode("overwrite").saveAsTable("lh_nyc_data.dbo.silver_taxi_zones")

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 9, Finished, Available, Finished, False)

In [ ]:
#Checking and fixing taxi data
taxi_data_examples = {}

for year in [2023,2024,2025]:
    path = f"Files/bronze/nyc_taxi/{year}/01/yellow_tripdata_{year}-01.parquet"

    taxi_data_examples[year] = spark.read.parquet(path)
    print(year)
    taxi_data_examples[year].printSchema()



StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 10, Finished, Available, Finished, False)

2023
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

2024
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropo

In [ ]:
display(taxi_data_examples[2025].limit(10))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f251f195-7cc0-4131-a42c-04ea49078fe4)

In [ ]:
#loading all taxi data
taxi_data = {}

for year in [2023,2024,2025]:
    for month in range(1, 13):
        path = f"Files/bronze/nyc_taxi/{year}/{month:02d}/yellow_tripdata_{year}-{month:02d}.parquet"
        taxi_data[f"{year}-{month:02d}"] = spark.read.parquet(path)

print(len(taxi_data))




StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 12, Finished, Available, Finished, False)

36


In [ ]:
for month in taxi_data:
    part = taxi_data[month]
    for column in part.columns:
        part = part.withColumnRenamed(column, column.lower())

taxi_data[month] = part

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 13, Finished, Available, Finished, False)

In [ ]:
print(taxi_data["2025-01"].columns)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 14, Finished, Available, Finished, False)

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


In [ ]:
print(taxi_data["2023-01"].columns)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 15, Finished, Available, Finished, False)

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


In [ ]:
for month in taxi_data:
    cells_data = taxi_data[month]

    cells_data = cells_data.withColumn("vendorid", col("vendorid").cast("int"))
    cells_data = cells_data.withColumn("pulocationid", col("pulocationid").cast("int"))
    cells_data = cells_data.withColumn("dolocationid", col("dolocationid").cast("int"))
    cells_data = cells_data.withColumn("passenger_count", col("passenger_count").cast("double"))
    cells_data = cells_data.withColumn("ratecodeid", col("ratecodeid").cast("double"))

    taxi_data[month] = cells_data

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 16, Finished, Available, Finished, False)

In [ ]:
print(taxi_data["2023-01"].printSchema())
print(taxi_data["2024-01"].printSchema())
print(taxi_data["2025-01"].printSchema())

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 17, Finished, Available, Finished, False)

root
 |-- vendorid: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecodeid: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: integer (nullable = true)
 |-- dolocationid: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

None
root
 |-- vendorid: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_d

In [ ]:
#uniting tables
months = list(taxi_data)

df_taxi = taxi_data[months[0]]

for month in months[1:]:
    df_taxi = df_taxi.unionByName(
        taxi_data[month],
        allowMissingColumns=True
    )

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 18, Finished, Available, Finished, False)

In [ ]:
#checking united data
print(df_taxi.count())
df_taxi.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "total_amount").summary("count", "min","max").show()

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 19, Finished, Available, Finished, False)

128202548
+-------+-------------+------------+
|summary|trip_distance|total_amount|
+-------+-------------+------------+
|  count|    128202548|   128202548|
|    min|          0.0|    -2265.45|
|    max|    398608.62|   863380.37|
+-------+-------------+------------+



In [ ]:
df_taxi.agg(
    F.min("tpep_pickup_datetime").alias("first_pickup"),
    F.max("tpep_pickup_datetime").alias("last_pickup"),
    F.min("tpep_dropoff_datetime").alias("first_dropoff"),
    F.max("tpep_dropoff_datetime").alias("last_dropoff"),
    F.count("tpep_pickup_datetime").alias("filled_pickup"),
    F.count("tpep_dropoff_datetime").alias("filled_dropoff")
).show(truncate=False)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 20, Finished, Available, Finished, False)

+-------------------+-------------------+-------------------+-------------------+-------------+--------------+
|first_pickup       |last_pickup        |first_dropoff      |last_dropoff       |filled_pickup|filled_dropoff|
+-------------------+-------------------+-------------------+-------------------+-------------+--------------+
|2001-01-01 00:06:49|2026-06-26 23:53:12|1970-01-20 10:16:32|2026-06-27 20:59:10|128202548    |128202548     |
+-------------------+-------------------+-------------------+-------------------+-------------+--------------+



In [ ]:
df_taxi = df_taxi.filter(
    F.col("tpep_pickup_datetime") >= "2023-01-01"


)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 21, Finished, Available, Finished, False)

In [ ]:
df_taxi = df_taxi.filter(
    (F.col("tpep_pickup_datetime") < "2026-01-01") &
    (F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
)



StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 22, Finished, Available, Finished, False)

In [ ]:
df_taxi = df_taxi.withColumn(
    "duration_minutes",
    F.expr(
        "timestampdiff(SECOND, tpep_pickup_datetime, "
        "tpep_dropoff_datetime) / 60.0"
    )
)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 23, Finished, Available, Finished, False)

In [ ]:
display(
    df_taxi.select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "duration_minutes",
        "trip_distance",
        "total_amount"
    )
    .orderBy(F.col("duration_minutes").desc())
    .limit(10)
)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 495786aa-d9cb-4898-9314-a1165f8cc4a1)

In [ ]:
df_taxi = df_taxi.filter((F.col("duration_minutes") >= 0) &(F.col("duration_minutes") <= 1440))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 25, Finished, Available, Finished, False)

In [ ]:
df_taxi = df_taxi.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))

df_taxi = df_taxi.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 26, Finished, Available, Finished, False)

In [ ]:
df_taxi.agg(
    F.count(F.when(F.col("trip_distance") == 0, 1)).alias("zero_distance"),
    F.count(F.when(F.col("trip_distance") < 0, 1)).alias("negative_distance"),
    F.count(F.when(F.col("total_amount") == 0, 1)).alias("zero_amount"),
    F.count(F.when(F.col("total_amount") < 0, 1)).alias("negative_amount")
    
).show()

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 27, Finished, Available, Finished, False)

+-------------+-----------------+-----------+---------------+
|zero_distance|negative_distance|zero_amount|negative_amount|
+-------------+-----------------+-----------+---------------+
|      2952380|                0|      17795|        1959860|
+-------------+-----------------+-----------+---------------+



In [ ]:
display(
    df_taxi.filter(F.col("total_amount") < 0)
    .select(
        "payment_type",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "trip_distance",
        "duration_minutes"
    )
    .limit(10)
)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, caee736c-1ec2-44b9-b78a-d73f742e1e29)

In [ ]:
display(
    df_taxi.select(
        "trip_distance",
        "duration_minutes",
        "total_amount",
        "ratecodeid"
    )
    .orderBy(F.col("trip_distance").desc())
    .limit(10)
)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 718595b1-a91c-4164-a96f-0b927acafcbe)

In [ ]:
print(df_taxi.filter(F.col("trip_distance") > 1000).count())

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 31, Finished, Available, Finished, False)

3973


In [ ]:
df_taxi = df_taxi.withColumn(
    "trip_distance",
    F.when(
        F.col("trip_distance") > 1000,
        F.lit(None).cast("double")
    ).otherwise(F.col("trip_distance"))
)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 32, Finished, Available, Finished, False)

In [ ]:
display(df_taxi.groupBy("passenger_count").count().orderBy("passenger_count"))

display(df_taxi.groupBy("ratecodeid").count().orderBy("ratecodeid"))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a8470d92-f431-4a78-931d-c01c639d8558)

SynapseWidget(Synapse.DataFrame, 581f25c4-e67b-4596-982c-9f33a935b5dc)

In [ ]:
df_taxi = df_taxi.withColumn("passenger_count", F.col("passenger_count").cast("int"))
df_taxi = df_taxi.withColumn("ratecodeid", F.col("ratecodeid").cast("int"))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 34, Finished, Available, Finished, False)

In [ ]:
zones = spark.table("lh_nyc_data.dbo.silver_taxi_zones")

zone_ids = zones.select(F.col("LocationID").alias("zone_id"))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 36, Finished, Available, Finished, False)

In [ ]:
zones = spark.table("lh_nyc_data.dbo.silver_taxi_zones")

pickup_ids = df_taxi.select("pulocationid").distinct()
dropoff_ids = df_taxi.select("dolocationid").distinct()

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 38, Finished, Available, Finished, False)

In [ ]:
print(df_taxi.filter(
        (F.abs(F.col("total_amount")) > 10000) |
        (F.abs(F.col("fare_amount")) > 10000)
    ).count())

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 42, Finished, Available, Finished, False)

14


In [ ]:
df_taxi = df_taxi.filter(
    (F.abs(F.col("total_amount")) <= 10000) &
    (F.abs(F.col("fare_amount")) <= 10000))
    

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 43, Finished, Available, Finished, False)

In [ ]:
rows = df_taxi.count()
unique_rows = df_taxi.dropDuplicates().count()

print("all rows", rows)
print("repeated", rows - unique_rows)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 44, Finished, Available, Finished, False)

all rows 128195308
repeated 7


In [ ]:
df_taxi = df_taxi.dropDuplicates()

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 45, Finished, Available, Finished, False)

In [ ]:
df_taxi.write.format("delta").mode("overwrite").saveAsTable("lh_nyc_data.dbo.silver_taxi_trips")

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 46, Finished, Available, Finished, False)

In [ ]:
saved_taxi = spark.table("lh_nyc_data.dbo.silver_taxi_trips")
print(saved_taxi.count())

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 47, Finished, Available, Finished, False)

128195301


----------------------

In [12]:
df_air = spark.table("lh_nyc_data.dbo.bronze_openaq_measurements")
df_air.printSchema()

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 16, Finished, Available, Finished, False)

root
 |-- sensor_id: long (nullable = true)
 |-- location_id: long (nullable = true)
 |-- station: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- request_month: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingested_at_utc: string (nullable = true)
 |-- value: double (nullable = true)
 |-- flagInfo.hasFlags: boolean (nullable = true)
 |-- parameter.id: long (nullable = true)
 |-- parameter.name: string (nullable = true)
 |-- parameter.units: string (nullable = true)
 |-- parameter.displayName: string (nullable = true)
 |-- period.label: string (nullable = true)
 |-- period.interval: string (nullable = true)
 |-- period.datetimeFrom.utc: string (nullable = true)
 |-- period.datetimeFrom.local: string (nullable = true)
 |-- period.datetimeTo.utc: string (nullable = true)
 |-- period.datetimeTo.local: string (nullable = true)
 |-- coverage.expectedCount: long (nullable = true)
 |-- coverage.expectedInterval: string (nullable = true)
 |-- 

In [13]:
df_air = df_air.withColumnRenamed("parameter.units", "units")
df_air = df_air.withColumnRenamed("period.interval", "interval")
df_air = df_air.withColumnRenamed("period.datetimeFrom.utc", "datetime_from_utc")
df_air = df_air.withColumnRenamed("period.datetimeFrom.local", "datetime_from_local")
df_air = df_air.withColumnRenamed("flagInfo.hasFlags", "has_flags")

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 17, Finished, Available, Finished, False)

In [14]:
display(df_air.select(
        "sensor_id",
        "station",
        "value",
        "units",
        "interval",
        "datetime_from_utc",
        "datetime_from_local",
        "has_flags"
    ).limit(10))


StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3d6150b0-371e-45c3-ba4b-1f9bd663189d)

In [15]:
print("all rows", df_air.count())
print("no values", df_air.filter(col("value").isNull()).count())
print("negative", df_air.filter(col("value") < 0).count())
print("with flag", df_air.filter(col("has_flags") == True).count())


StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 19, Finished, Available, Finished, False)

all rows 119073
no values 2817
negative 0
with flag 4210


In [16]:
df_air = df_air.withColumn("measured_at_utc",F.to_timestamp("datetime_from_utc"))

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 20, Finished, Available, Finished, False)

In [17]:
print("issue with date",df_air.filter(col("measured_at_utc").isNull()).count())

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 21, Finished, Available, Finished, False)

issue with date 0


In [18]:
df_air = df_air.withColumn("measured_at_local", F.from_utc_timestamp("measured_at_utc", "America/New_York"))
df_air = df_air.withColumn("measurement_date", F.to_date("measured_at_local"))
display(df_air.select("datetime_from_local", "measured_at_local", "measurement_date").limit(5))

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 254e3ba3-4ab3-4b1f-a8d8-2f198dcd3b8d)

In [19]:
#dublicates of the sensors
duplicates = df_air.groupBy("sensor_id", "measured_at_utc", "interval").count()
display(duplicates.filter(col("count") > 1))

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c740446-ddc3-42ef-afac-ec9295772d56)

In [20]:
df_air.groupBy("units", "interval").count().show()

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 24, Finished, Available, Finished, False)

+-----+--------+------+
|units|interval| count|
+-----+--------+------+
|µg/m³|01:00:00|119073|
+-----+--------+------+



In [21]:
for column in df_air.columns:
    df_air = df_air.withColumnRenamed(column, column.replace(".", "_"))

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 25, Finished, Available, Finished, False)

In [22]:
df_air.write.format("delta").mode("overwrite").saveAsTable("lh_nyc_data.dbo.silver_openaq_measurements")
print(spark.table("lh_nyc_data.dbo.silver_openaq_measurements").count())

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 26, Finished, Available, Finished, False)

119073


In [23]:
path = "/lakehouse/default/Files/bronze/epa/no2/daily_42602_2023.zip"
epa_2023 = pd.read_csv(
    path,
    dtype={"State Code": str, "County Code": str, "Site Num": str})


print(epa_2023.columns.tolist())
display(epa_2023.head(3))

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 27, Finished, Available, Finished, False)

['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Sample Duration', 'Pollutant Standard', 'Date Local', 'Units of Measure', 'Event Type', 'Observation Count', 'Observation Percent', 'Arithmetic Mean', '1st Max Value', '1st Max Hour', 'AQI', 'Method Code', 'Method Name', 'Local Site Name', 'Address', 'State Name', 'County Name', 'City Name', 'CBSA Name', 'Date of Last Change']


SynapseWidget(Synapse.DataFrame, 89ce58fa-bf94-46c3-84c3-fba7f3cea1d9)

In [24]:
nyc_2023 = epa_2023[
    (epa_2023["State Code"] == "36") &
    (epa_2023["County Code"].isin(["005", "047", "061", "081", "085"]))]

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 28, Finished, Available, Finished, False)

In [25]:
nyc_2023 = epa_2023[
    (epa_2023["State Code"] == "36") &
    (epa_2023["County Code"].isin(["005", "047", "061", "081", "085"]))]

columns = [
    "County Name",
    "Local Site Name",
    "Sample Duration",
    "Pollutant Standard",
    "Units of Measure"
]

display(nyc_2023[columns].drop_duplicates())

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 67cd7bf0-44c7-4d7e-aaaf-04b6b6702936)

In [26]:
epa_parts = []

for year in [2023, 2024, 2025]:
    path = f"/lakehouse/default/Files/bronze/epa/no2/daily_42602_{year}.zip"

    part = pd.read_csv(
        path,
        dtype={"State Code": str, "County Code": str, "Site Num": str}
    )

    part = part[
        (part["State Code"] == "36") &
        (part["County Code"].isin(["005", "047", "061", "081", "085"]))
    ]

    epa_parts.append(part)

epa_nyc = pd.concat(epa_parts, ignore_index=True)

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 30, Finished, Available, Finished, False)

In [27]:
display(epa_nyc[columns].drop_duplicates())
print("all rows", len(epa_nyc))

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9b99bedc-a10b-4dc8-b59f-cd546d262449)

all rows 4243


In [28]:
epa_nyc["Date Local"] = pd.to_datetime(epa_nyc["Date Local"], errors="coerce")

print("No date", epa_nyc["Date Local"].isna().sum())
print("empty", epa_nyc["Arithmetic Mean"].isna().sum())
print("Negative", (epa_nyc["Arithmetic Mean"] < 0).sum())
print("First Date", epa_nyc["Date Local"].min())
print("Last Date", epa_nyc["Date Local"].max())

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 32, Finished, Available, Finished, False)

No date 0
empty 0
Negative 0
First Date 2023-01-01 00:00:00
Last Date 2025-12-31 00:00:00


In [29]:
keys = ["State Code", "County Code", "Site Num", "POC", "Date Local"]
print("Dublicates", epa_nyc.duplicated(subset=keys).sum())

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 34, Finished, Available, Finished, False)

Dublicates 0


In [30]:
epa_nyc.columns = epa_nyc.columns.str.lower().str.replace(" ", "_", regex=False)
df_no2 = spark.createDataFrame(epa_nyc)
df_no2 = df_no2.withColumn("date_local", F.to_date("date_local"))

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 35, Finished, Available, Finished, False)

In [31]:
df_no2.printSchema()

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 36, Finished, Available, Finished, False)

root
 |-- state_code: string (nullable = true)
 |-- county_code: string (nullable = true)
 |-- site_num: string (nullable = true)
 |-- parameter_code: long (nullable = true)
 |-- poc: long (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- datum: string (nullable = true)
 |-- parameter_name: string (nullable = true)
 |-- sample_duration: string (nullable = true)
 |-- pollutant_standard: string (nullable = true)
 |-- date_local: date (nullable = true)
 |-- units_of_measure: string (nullable = true)
 |-- event_type: void (nullable = true)
 |-- observation_count: long (nullable = true)
 |-- observation_percent: double (nullable = true)
 |-- arithmetic_mean: double (nullable = true)
 |-- 1st_max_value: double (nullable = true)
 |-- 1st_max_hour: long (nullable = true)
 |-- aqi: long (nullable = true)
 |-- method_code: long (nullable = true)
 |-- method_name: string (nullable = true)
 |-- local_site_name: string (nullable = true)
 |-- add

In [32]:
df_no2 = df_no2.withColumn("event_type", col("event_type").cast("string"))
df_no2 = df_no2.withColumn("date_of_last_change", F.to_date("date_of_last_change"))

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 37, Finished, Available, Finished, False)

In [33]:
df_no2.write.format("delta").mode("overwrite").saveAsTable("lh_nyc_data.dbo.silver_epa_no2_daily")

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 38, Finished, Available, Finished, False)

In [34]:
print(spark.table("lh_nyc_data.dbo.silver_epa_no2_daily").count())

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 39, Finished, Available, Finished, False)

4243


----------------------------------------------------------------------

In [35]:
df_fx = spark.table("lh_nyc_data.dbo.`D USD EUR SP00`")

df_fx = df_fx.filter(col("TIME_PERIOD") >= "2023-01-01")
df_fx = df_fx.filter(col("TIME_PERIOD") < "2026-01-01")

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 40, Finished, Available, Finished, False)

In [36]:
df_fx = df_fx.select("TIME_PERIOD", "OBS_VALUE")

df_fx = df_fx.withColumnRenamed("TIME_PERIOD", "rate_date")
df_fx = df_fx.withColumnRenamed("OBS_VALUE", "usd_per_eur")

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 41, Finished, Available, Finished, False)

In [37]:
display(df_fx.limit(5))
print(df_fx.count())

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8b897c3d-29c2-4ac5-a58b-f36be4300e83)

766


In [39]:
df_fx.write.format("delta").mode("overwrite").saveAsTable("lh_nyc_data.dbo.silver_fx_rates")

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 44, Finished, Available, Finished, False)

In [40]:
df_gdp = spark.table("lh_nyc_data.dbo.bronze_worldbank_gdp")

df_gdp = df_gdp.filter(col("countryiso3code") == "USA")
df_gdp = df_gdp.filter(col("date") >= 2023)
df_gdp = df_gdp.filter(col("date") <= 2025)

df_gdp = df_gdp.select("countryiso3code", "date", "value")

df_gdp = df_gdp.withColumnRenamed("countryiso3code", "country_code")
df_gdp = df_gdp.withColumnRenamed("date", "year")
df_gdp = df_gdp.withColumnRenamed("value", "gdp_usd")

display(df_gdp)

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5f5d1789-448a-482c-a3f5-421de1ae135c)

In [41]:
df_gdp.write.format("delta").mode("overwrite").saveAsTable("lh_nyc_data.dbo.silver_gdp")

StatementMeta(, a53bc1e9-c64b-444b-9993-df20ef5f0073, 46, Finished, Available, Finished, False)